In [1]:
import logging
import logging.config
import os
from pathlib import Path

import httpx
import pandas as pd

from graal.summary.blind_eval_project import BlindEvalProject
from graal.summary.llm_clients import AlbertAPIClient, ChatGPTAPIClient
from graal.utils.amendment_pre_processor import AmendmentPreProcessor

logging.config.fileConfig("logging.conf")


In [ ]:
# MAIN

DATA_FOLDER = os.getenv("DATA_FOLDER", "data")
METRICS = ["Correct", "Complet", "Concis"]

COLUMN_ORDER = ["ID", "Exposé amdt", "Corps amdt"]
for i in range(1, 3):
    COLUMN_ORDER.append(f"Objet {i}")
    for metric in METRICS:
        COLUMN_ORDER.append(f"{i} - {metric}")

EXCEL_OUTPUT_FILE = Path(f"{DATA_FOLDER}/blind_eval_summaries/eval_aveugle_objets.xlsx")
PROJECT_SAVE_LOCATION = Path(
    f"{DATA_FOLDER}/blind_eval_summaries/project_eval_aveugle_objets.pkl"
)
GRAAL_CONFIG_FILE = Path(
    f"{DATA_FOLDER}/config_graal/Fichier de configuration GRAAL - DSS - 28 Nov 2024.xlsx"
)
FILE_TO_SAMPLE_FROM = Path(
    f"{DATA_FOLDER}/exports_lectures/PLFSS 2025/BDD_AN_L1_SP_Amendements_copie_valeurs.xlsx"
)

attribution_mappings_excel = pd.read_excel(GRAAL_CONFIG_FILE, sheet_name=None)

try:
    project = BlindEvalProject.load_from_disk(PROJECT_SAVE_LOCATION)
    logging.info(f"Project '{PROJECT_SAVE_LOCATION}' successfully loaded")
except (FileNotFoundError, EOFError):
    logging.info(f"Creating new project '{PROJECT_SAVE_LOCATION}'")
    amendments_df = AmendmentPreProcessor.load_amendments_excel(
        input_files=[FILE_TO_SAMPLE_FROM]
    )
    amendments_df = AmendmentPreProcessor.remap_columns_in_json_amendments(
        amendments_df
    )

    amendments_df = amendments_df[amendments_df["Objet amdt"].str.strip() != ""]
    amendments_df = amendments_df[
        ~amendments_df["Objet amdt"].str.contains(
            "Amendement rédactionnel|Supprimer cet article|Supprimer l'article|Amendement de coordination|Modifier l'alinea|Modifier la rédaction|irr\?",
            na=False,
        )
    ]

    project = BlindEvalProject(
        amendments_df=amendments_df,
        metrics=METRICS,
        config_prompt=attribution_mappings_excel["Prompt Objet"].to_string(),
        rate_limiting_config={"albert": 10},
    )

llama_70B_client = AlbertAPIClient(  # noqa: N816
    base_url=httpx.URL(os.environ["ETALAB_BASE_URL"]),
    api_key=os.environ["ETALAB_API_KEY"],
    model_name=os.environ["ETALAB_MODEL_NAME"],
)

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
gpt_4_turbo_client = ChatGPTAPIClient(model_name="gpt-4-turbo", api_key=OPENAI_API_KEY)

gpt_4o_mini_client = ChatGPTAPIClient(model_name="gpt-4o-mini", api_key=OPENAI_API_KEY)

llm_clients = {
    "llama-3.1-70b-instruct": llama_70B_client,
    "gpt-4-turbo": gpt_4_turbo_client,
    "gpt-4o-mini": gpt_4o_mini_client,
}

project.add_next_n_rows(48, llm_clients)

project.to_excel(
    output_file=EXCEL_OUTPUT_FILE, column_order=COLUMN_ORDER, excluded_ids=[]
)
project.dump_to_disk(output_file=PROJECT_SAVE_LOCATION)
# Open the Excel file
os.system(f'open "{EXCEL_OUTPUT_FILE}"')  # noqa: S605
project.mapping_obj_to_author